In [1]:
import numpy as np
from pathlib import Path
import sys

PROJECT_DIR = str(Path.cwd().parent) + "/"
sys.path.append(PROJECT_DIR)

from mpp_project.end_game_solver import solve_endgame_dp

print("🌌 DÉMARRAGE DU LABORATOIRE DP : PROFONDEUR TEMPORELLE (4 MATCHS)")

# ==========================================
# 1. CRÉATION DES 4 MATCHS "PIPEAU" (t=28 à 31)
# ==========================================
N_MATCHS = 4
stop_t = 32 - N_MATCHS # 28

true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float32) 
crowd = np.array([0.70, 0.20, 0.10], dtype=np.float32)
# On choisit des gains qui donnent des écarts pairs (pour éviter les arrondis dans la grille / 2.0)
gains = np.array([20, 50, 90], dtype=np.int32) 

match_probs = np.zeros((32, 3), dtype=np.float32)
crowds = np.zeros((32, 3), dtype=np.float32)
gains_1N2 = np.zeros((32, 3), dtype=np.int32)

for t in range(stop_t, 32):
    match_probs[t] = true_proba
    crowds[t] = crowd
    gains_1N2[t] = gains

p_empirique_1D = np.zeros((32, 3, 250), dtype=np.float32)
p_empirique_1D[:, :, 0] = 1.0 # Peloton désactivé (fantôme)

alphas = np.ones(32, dtype=np.float32)
roles_mock = np.full(32, -1, dtype=np.int32)

V_term = np.zeros((501, 501, 2, 2, 2, 2), dtype=np.float32)
for g1 in range(501):
    for g2 in range(501):
        if (g1 - 250) >= 0 and (g2 - 250) >= 0:
            V_term[g1, g2, :, :, :, :] = 1.0

# ==========================================
# 2. LE SOLVEUR THÉORIQUE EXACT (RÉCURSIF)
# ==========================================
# Cette fonction calcule l'arbre combinatoire complet sans utiliser de grille
def exact_theoretical_wr(t, current_gap):
    if t == 32:
        return 1.0 if current_gap >= 0 else 0.0

    max_wr_for_this_match = 0.0
    
    # On teste les 3 choix possibles pour l'Agent
    for my_action in range(3):
        wr_action = 0.0
        
        # On itère sur ce qui se passe VRAIMENT sur le terrain
        for actual_out in range(3):
            p_out = true_proba[actual_out]
            if p_out == 0: continue
            
            my_gain = gains[actual_out] if my_action == actual_out else 0
            
            # On itère sur les choix possibles de BOB
            for bob_action in range(3):
                p_bob = crowd[bob_action]
                if p_bob == 0: continue
                
                bob_gain = gains[actual_out] if bob_action == actual_out else 0
                
                # Nouveau gap après ce match
                new_gap = current_gap + my_gain - bob_gain
                
                # On plonge dans le futur (match suivant)
                wr_action += p_out * p_bob * exact_theoretical_wr(t + 1, new_gap)
                
        # L'Agent est intelligent : il prendra l'action qui maximise son WR futur
        if wr_action > max_wr_for_this_match:
            max_wr_for_this_match = wr_action
            
    return max_wr_for_this_match

# ==========================================
# 3. EXÉCUTION DE L'ORACLE NUMBA (DP)
# ==========================================
print(f"🧠 Lancement de la DP (Sur {N_MATCHS} matchs)...")
V_history = solve_endgame_dp(
    match_probs, crowds, gains_1N2, p_empirique_1D, alphas, 
    roles_mock, roles_mock, roles_mock, V_term, stop_t=stop_t
)

# On extrait la décision à t=28 (Il reste 4 matchs)
Q_table_t28 = V_history[stop_t]

# ==========================================
# 4. LES TESTS (Confrontation des deux mondes)
# ==========================================
print("\n" + "="*50)
print(f"📊 RÉSULTATS DES TESTS (Horizon = {N_MATCHS} matchs)")
print("="*50)

idx_g2_safe = 500  

def test_scenario_multiple(nom, gap_initial):
    # Calcul DP (Lecture instantanée dans la grille générée)
    gap_grid = int(gap_initial / 2.0)
    idx_g1 = gap_grid + 250
    wr_dp = Q_table_t28[idx_g1, idx_g2_safe, 0, 0, 0, 0]
    
    # Calcul Théorique (Parcours de l'arbre combinatoire)
    wr_th = exact_theoretical_wr(stop_t, gap_initial)
    
    print(f"▶️ TEST : {nom}")
    print(f"   Retard sur Bob : {-gap_initial} pts")
    print(f"   WR Théorique   : {wr_th*100:05.2f}%")
    print(f"   WR Calculé DP  : {wr_dp*100:05.2f}%")
    
    if abs(wr_dp - wr_th) < 0.001:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)


max_gain_possible = N_MATCHS * 90 # 4 * 90 = 360

# --- SCÉNARIO 1 : MISSION IMPOSSIBLE ---
# Même si on gagne 4 fois 90 pts (360) et que Bob prend 0, on ne comble pas le trou.
test_scenario_multiple("Mission Impossible absolue", gap_initial=-(max_gain_possible + 20))

# --- SCÉNARIO 2 : LE GRAND CHELEM OBLIGATOIRE ---
# On a 350 pts de retard. 
# 3 Outsiders (270) + 1 Nul (50) = 320 (Insuffisant).
# Il faut EXACTEMENT 4 Outsiders à 90 pts (360), ET que Bob fasse des erreurs.
test_scenario_multiple("Le Grand Chelem (4 Outsiders Requis)", gap_initial=-350)

# --- SCÉNARIO 3 : LE DROIT À L'ERREUR ---
# On a 300 pts de retard. 
# 4 Outsiders ça passe (360).
# 3 Outsiders (270) + 1 Nul (50) = 320 -> Ça passe !
# 3 Outsiders (270) + 1 Favori (20) = 290 -> Insuffisant.
# L'arbre combinatoire devient complexe, mais les deux algorithmes doivent être alignés.
test_scenario_multiple("Le Droit à l'Erreur (3 Outsiders + 1 Nul suffisent)", gap_initial=-300)

# --- SCÉNARIO 4 : L'OUTSIDER DÈS LE PREMIER MATCH ? ---
# Test intermédiaire : retard gérable, on veut vérifier la stratégie adaptative.
test_scenario_multiple("Retard modéré (Stratégie Mixte)", gap_initial=-150)

# --- SCÉNARIO 5 : ON GÈRE L'AVANCE ---
# On est devant. On couvre mathématiquement pendant 4 matchs.
test_scenario_multiple("Gestion de l'avance pendant 4 matchs", gap_initial=50)

🌌 DÉMARRAGE DU LABORATOIRE DP : PROFONDEUR TEMPORELLE (4 MATCHS)
🧠 Lancement de la DP (Sur 4 matchs)...

📊 RÉSULTATS DES TESTS (Horizon = 4 matchs)
▶️ TEST : Mission Impossible absolue
   Retard sur Bob : 380 pts
   WR Théorique   : 00.00%
   WR Calculé DP  : 00.00%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Le Grand Chelem (4 Outsiders Requis)
   Retard sur Bob : 350 pts
   WR Théorique   : 00.03%
   WR Calculé DP  : 00.03%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Le Droit à l'Erreur (3 Outsiders + 1 Nul suffisent)
   Retard sur Bob : 300 pts
   WR Théorique   : 00.05%
   WR Calculé DP  : 00.05%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Retard modéré (Stratégie Mixte)
   Retard sur Bob : 150 pts
   WR Théorique   : 07.29%
   WR Calculé DP  : 07.29%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Gestion de l'avance pendant 4 matchs
   Retard sur Bob

In [2]:
print("🚀 DÉMARRAGE DU LABORATOIRE DP : PROFONDEUR TEMPORELLE + BOOSTER x2")

# ==========================================
# 1. CRÉATION DES 4 MATCHS "PIPEAU" (t=28 à 31)
# ==========================================
N_MATCHS = 4
stop_t = 32 - N_MATCHS # 28

true_proba = np.array([0.60, 0.25, 0.15], dtype=np.float32) 
crowd = np.array([0.70, 0.20, 0.10], dtype=np.float32)
gains = np.array([20, 50, 90], dtype=np.int32) 

match_probs = np.zeros((32, 3), dtype=np.float32)
crowds = np.zeros((32, 3), dtype=np.float32)
gains_1N2 = np.zeros((32, 3), dtype=np.int32)

for t in range(stop_t, 32):
    match_probs[t] = true_proba
    crowds[t] = crowd
    gains_1N2[t] = gains

p_empirique_1D = np.zeros((32, 3, 250), dtype=np.float32)
p_empirique_1D[:, :, 0] = 1.0 # Peloton désactivé (fantôme)

alphas = np.ones(32, dtype=np.float32)
roles_mock = np.full(32, -1, dtype=np.int32)

V_term = np.zeros((501, 501, 2, 2, 2, 2), dtype=np.float32)
for g1 in range(501):
    for g2 in range(501):
        if (g1 - 250) >= 0 and (g2 - 250) >= 0:
            V_term[g1, g2, :, :, :, :] = 1.0

# ==========================================
# 2. LE SOLVEUR THÉORIQUE EXACT (RÉCURSIF + BOOSTER)
# ==========================================
def exact_theoretical_wr_with_booster(t, current_gap, has_booster):
    if t == 32:
        return 1.0 if current_gap >= 0 else 0.0

    max_wr_for_this_match = 0.0
    
    # L'Agent teste ses 3 choix possibles
    for my_action in range(3):
        # L'Agent teste de jouer SANS ou AVEC le booster (s'il l'a)
        options_booster = [False, True] if has_booster else [False]
        
        for use_booster_now in options_booster:
            wr_action = 0.0
            
            # On itère sur ce qui se passe VRAIMENT sur le terrain
            for actual_out in range(3):
                p_out = true_proba[actual_out]
                if p_out == 0: continue
                
                base_my_gain = gains[actual_out] if my_action == actual_out else 0
                my_gain = base_my_gain * 2 if use_booster_now else base_my_gain
                
                # On itère sur les choix de BOB
                for bob_action in range(3):
                    p_bob = crowd[bob_action]
                    if p_bob == 0: continue
                    
                    bob_gain = gains[actual_out] if bob_action == actual_out else 0
                    
                    # Transition d'état
                    new_gap = current_gap + my_gain - bob_gain
                    next_has_booster = False if use_booster_now else has_booster
                    
                    # Plongée récursive dans le futur
                    wr_action += p_out * p_bob * exact_theoretical_wr_with_booster(t + 1, new_gap, next_has_booster)
                    
            if wr_action > max_wr_for_this_match:
                max_wr_for_this_match = wr_action
                
    return max_wr_for_this_match

# ==========================================
# 3. EXÉCUTION DE L'ORACLE NUMBA (DP)
# ==========================================
print(f"🧠 Lancement de la DP (Sur {N_MATCHS} matchs)...")
V_history = solve_endgame_dp(
    match_probs, crowds, gains_1N2, p_empirique_1D, alphas, 
    roles_mock, roles_mock, roles_mock, V_term, stop_t=stop_t
)

# On extrait la décision à t=28
Q_table_t28 = V_history[stop_t]

# ==========================================
# 4. LES TESTS (Confrontation des deux mondes)
# ==========================================
print("\n" + "="*50)
print(f"📊 RÉSULTATS DES TESTS (Horizon = {N_MATCHS} matchs | Base vs x2)")
print("="*50)

idx_g2_safe = 500  

def test_scenario_booster_multiple(nom, gap_initial):
    gap_grid = int(gap_initial / 2.0)
    idx_g1 = gap_grid + 250
    
    # --- SANS BOOSTER (Dimension 0) ---
    wr_dp_sans = Q_table_t28[idx_g1, idx_g2_safe, 0, 0, 0, 0]
    wr_th_sans = exact_theoretical_wr_with_booster(stop_t, gap_initial, has_booster=False)
    
    # --- AVEC BOOSTER (Dimension 1) ---
    wr_dp_avec = Q_table_t28[idx_g1, idx_g2_safe, 1, 0, 0, 0]
    wr_th_avec = exact_theoretical_wr_with_booster(stop_t, gap_initial, has_booster=True)
    
    print(f"▶️ TEST : {nom}")
    print(f"   Retard sur Bob : {-gap_initial} pts")
    print(f"   [Base] Théorie: {wr_th_sans*100:05.2f}% | DP: {wr_dp_sans*100:05.2f}%")
    print(f"   [ x2 ] Théorie: {wr_th_avec*100:05.2f}% | DP: {wr_dp_avec*100:05.2f}%")
    
    reussi = abs(wr_dp_sans - wr_th_sans) < 0.001 and abs(wr_dp_avec - wr_th_avec) < 0.001
    if reussi:
        print("   ✅ TEST RÉUSSI")
    else:
        print("   ❌ ÉCHEC DU TEST")
    print("-" * 50)


# Le gain maximum théorique en 4 matchs sans booster : 4 * 90 = 360
# Le gain maximum théorique en 4 matchs AVEC booster : (3 * 90) + (1 * 180) = 450

# --- SCÉNARIO 1 : MISSION IMPOSSIBLE (MÊME AVEC BOOSTER) ---
# On a 460 points de retard. Même l'enchaînement parfait de l'univers ne suffit pas.
test_scenario_booster_multiple("Désespoir absolu", gap_initial=-460)

# --- SCÉNARIO 2 : LE BOOSTER EST LA SEULE ISSUE ---
# On a 400 points de retard. Sans booster, max 360 pts -> on perd à coup sûr.
# Avec booster, max 450 pts -> une infime chance de gagner existe.
test_scenario_booster_multiple("Le Booster de la dernière chance", gap_initial=-400)

# --- SCÉNARIO 3 : LE DROIT À L'ERREUR ÉLARGI ---
# On a 300 points de retard.
# Sans booster, il faut un quasi sans-faute (ex: 3 Outsiders + 1 Nul).
# Avec booster, la marge d'erreur s'agrandit considérablement, le WR devrait exploser.
test_scenario_booster_multiple("Le Droit à l'erreur (Boost de probabilité)", gap_initial=-300)

# --- SCÉNARIO 4 : L'OPTIMISATION DU RISQUE ---
# On a 100 points de retard.
# Les deux mondes (avec et sans) ont d'excellentes chances, 
# mais l'arbre du Booster va explorer des centaines de "moments" parfaits pour se déclencher (sur un Nul ? sur un Outsider ?).
test_scenario_booster_multiple("Gestion stratégique du retard", gap_initial=-100)

# --- SCÉNARIO 5 : LE LEVIER SÉCURITAIRE ---
# On est devant de 50 points.
# Le booster ne sert plus à rattraper, mais à "tuer" l'espoir de Bob de nous doubler.
test_scenario_booster_multiple("L'Estocade du Leader", gap_initial=50)

🚀 DÉMARRAGE DU LABORATOIRE DP : PROFONDEUR TEMPORELLE + BOOSTER x2
🧠 Lancement de la DP (Sur 4 matchs)...

📊 RÉSULTATS DES TESTS (Horizon = 4 matchs | Base vs x2)
▶️ TEST : Désespoir absolu
   Retard sur Bob : 460 pts
   [Base] Théorie: 00.00% | DP: 00.00%
   [ x2 ] Théorie: 00.00% | DP: 00.00%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Le Booster de la dernière chance
   Retard sur Bob : 400 pts
   [Base] Théorie: 00.00% | DP: 00.00%
   [ x2 ] Théorie: 00.05% | DP: 00.05%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Le Droit à l'erreur (Boost de probabilité)
   Retard sur Bob : 300 pts
   [Base] Théorie: 00.05% | DP: 00.05%
   [ x2 ] Théorie: 00.97% | DP: 00.97%
   ✅ TEST RÉUSSI
--------------------------------------------------
▶️ TEST : Gestion stratégique du retard
   Retard sur Bob : 100 pts
   [Base] Théorie: 14.18% | DP: 14.18%
   [ x2 ] Théorie: 27.23% | DP: 27.23%
   ✅ TEST RÉUSSI
-----------------------------